# 04 — Baseline climate from CETRAD data

**Question:** What is the mean seasonal rainfall at OL JOGI FARM (Laikipia), and how does it compare to the flat defaults used in `03_trait_surface.ipynb`?

**Data:** CETRAD daily rainfall record, OL JOGI FARM station.  
**Method:** Fit dekadal α (mean storm depth) and λ (storm frequency) using `make_climate_parameters`, then compute mean seasonal rainfall over the 180-day growing season (dekads 7–24, ~March–August).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from farm.climate import make_climate_parameters, Climate

def despine(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

STATION   = 'OL JOGI FARM'
DATA_FILE = '../data/CETRAD/CETRAD_rainfall.csv'
INTERVAL  = 'dekad'
LGP       = 180   # length of growing period (days)
# Growing season = dekads 7–24 (0-indexed: 6:24), consistent with manuscript-figures.ipynb
SEAS_SLICE = slice(6, 24)   # 18 dekads × 10 days = 180 days

print('Setup complete.')

---
## 1 — Fit dekadal α and λ from CETRAD

In [ ]:
a_mid, l_mid, sd = make_climate_parameters(
    station=STATION,
    data_file=DATA_FILE,
    interval=INTERVAL
)

a_mid = np.array(a_mid)
l_mid = np.array(l_mid)

dekads = np.arange(1, len(a_mid) + 1)

df_dekad = pd.DataFrame({
    'dekad':  dekads,
    'alpha':  a_mid,
    'lambda': l_mid,
    'mean_daily_rf': a_mid * l_mid,
})

print(f'Number of dekads: {len(a_mid)}')
print()
print(df_dekad.to_string(index=False))

---
## 2 — Mean seasonal rainfall (growing season only)

In [ ]:
# Growing season subset (dekads 7–24)
a_seas = a_mid[SEAS_SLICE]
l_seas = l_mid[SEAS_SLICE]

# Mean daily RF per dekad × 10 days/dekad, summed over growing season
mean_seasonal_rf = np.sum(a_seas * l_seas * 10)

# Annual mean from full record
mean_annual_rf = np.sum(a_mid * l_mid * 10)

# Flat defaults used in 03_trait_surface.ipynb
alpha_default  = 10.0
lambda_default = 0.25
default_seasonal_rf = alpha_default * lambda_default * LGP

print('=== OL JOGI FARM (CETRAD, dekadal fit) ===')
print(f'  Mean annual rainfall:          {mean_annual_rf:.0f} mm/year')
print(f'  Mean seasonal RF (dekads 7–24): {mean_seasonal_rf:.0f} mm / {LGP} days')
print(f'  Mean daily RF in-season:        {mean_seasonal_rf/LGP:.2f} mm/day')
print()
print('=== 03_trait_surface.ipynb defaults (Climate()) ===')
print(f'  alpha_r = {alpha_default}, lambda_r = {lambda_default}')
print(f'  Mean seasonal RF ({LGP} days):    {default_seasonal_rf:.0f} mm')
print(f'  Mean daily RF:                  {alpha_default * lambda_default:.2f} mm/day')
print()
print(f'Difference: {mean_seasonal_rf - default_seasonal_rf:+.0f} mm  '
      f'({(mean_seasonal_rf/default_seasonal_rf - 1)*100:+.1f}%)')

---
## 3 — Plot: dekadal α, λ, and mean daily rainfall through the year

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
fig.subplots_adjust(hspace=0.08)

x = dekads
# Shade the growing season
for ax in axes:
    ax.axvspan(7, 24, alpha=0.10, color='steelblue', label='Growing season (dekads 7–24)')

# α
axes[0].bar(x, a_mid, color='steelblue', alpha=0.75)
axes[0].axhline(alpha_default, color='tomato', lw=1.5, ls='--', label=f'Default α={alpha_default}')
axes[0].set_ylabel('α  (mm/storm)', fontsize=10)
axes[0].set_title('(a)  Mean storm depth  α', fontsize=10, loc='left')
axes[0].legend(fontsize=8, frameon=False)
despine(axes[0])

# λ
axes[1].bar(x, l_mid, color='steelblue', alpha=0.75)
axes[1].axhline(lambda_default, color='tomato', lw=1.5, ls='--', label=f'Default λ={lambda_default}')
axes[1].set_ylabel('λ  (day⁻¹)', fontsize=10)
axes[1].set_title('(b)  Storm frequency  λ', fontsize=10, loc='left')
axes[1].legend(fontsize=8, frameon=False)
despine(axes[1])

# Mean daily RF = α × λ
axes[2].bar(x, a_mid * l_mid, color='steelblue', alpha=0.75)
axes[2].axhline(alpha_default * lambda_default, color='tomato', lw=1.5, ls='--',
                label=f'Default α×λ = {alpha_default * lambda_default:.2f} mm/day')
axes[2].set_ylabel('α × λ  (mm/day)', fontsize=10)
axes[2].set_xlabel('Dekad of year', fontsize=10)
axes[2].set_title('(c)  Mean daily rainfall  α × λ', fontsize=10, loc='left')
axes[2].legend(fontsize=8, frameon=False)
despine(axes[2])

fig.suptitle(f'OL JOGI FARM — dekadal rainfall parameters\n'
             f'Mean seasonal RF (dekads 7–24): {mean_seasonal_rf:.0f} mm  |  '
             f'Default: {default_seasonal_rf:.0f} mm',
             fontsize=11)

plt.savefig('../output/04_climate_dekadal.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4 — Stochastic check: simulated seasonal RF distribution

Run N=500 synthetic seasons using the site-fitted parameters and compare the distribution to the flat default.

In [ ]:
N = 500
climate_site    = Climate(alpha_r=a_mid.tolist(), lambda_r=l_mid.tolist())
climate_default = Climate(alpha_r=alpha_default, lambda_r=lambda_default)

# Simulate N seasons, collect total seasonal RF
def seasonal_rf_samples(climate_obj, n, lgp=180, start_dekad=6):
    """Return array of total growing-season rainfall for n synthetic years."""
    # Growing season starts at dekad 7 (0-indexed: 6), i.e. doy ~61
    doy_start = start_dekad * 10 + 1  # dekad 7 starts at doy 61
    samples = []
    for seed in range(n):
        np.random.seed(seed)
        rf = Climate.generate(climate_obj.alpha_r, climate_obj.lambda_r,
                              t_sim=lgp, doy_start=doy_start)
        samples.append(rf.sum())
    return np.array(samples)

rf_site    = seasonal_rf_samples(climate_site,    N)
rf_default = seasonal_rf_samples(climate_default, N)

print('Site-fitted climate (OL JOGI FARM, dekadal):')
print(f'  Mean: {rf_site.mean():.0f} mm  |  Median: {np.median(rf_site):.0f} mm  |  '
      f'IQR: [{np.percentile(rf_site,25):.0f}, {np.percentile(rf_site,75):.0f}] mm')
print()
print('Flat default (α=10, λ=0.25):')
print(f'  Mean: {rf_default.mean():.0f} mm  |  Median: {np.median(rf_default):.0f} mm  |  '
      f'IQR: [{np.percentile(rf_default,25):.0f}, {np.percentile(rf_default,75):.0f}] mm')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

bins = np.linspace(0, max(rf_site.max(), rf_default.max()) * 1.05, 40)

ax.hist(rf_site,    bins=bins, alpha=0.6, color='steelblue', label='Site-fitted (OL JOGI FARM)')
ax.hist(rf_default, bins=bins, alpha=0.6, color='tomato',    label='Flat default (α=10, λ=0.25)')

ax.axvline(rf_site.mean(),    color='steelblue', lw=2, ls='--')
ax.axvline(rf_default.mean(), color='tomato',    lw=2, ls='--')

ax.set_xlabel('Seasonal rainfall [mm]', fontsize=11)
ax.set_ylabel('Count  (N=500 seasons)', fontsize=11)
ax.set_title('Distribution of simulated seasonal rainfall\n'
             'Site-fitted vs. flat default', fontsize=11)
ax.legend(fontsize=9, frameon=False)
despine(ax)

plt.tight_layout()
plt.savefig('../output/04_climate_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| | Mean seasonal RF | Notes |
|---|---|---|
| OL JOGI FARM (CETRAD, dekadal) | see output above | Seasonally varying α and λ |
| `03_trait_surface.ipynb` default | ~450 mm | Flat α=10, λ=0.25 all season |

**Key question:** Are the means close enough that the flat defaults are an acceptable simplification, or does the within-season variation in storm regime matter for the trait surface results?